In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL = "BAAI/bge-m3"
DRIVE = Path("/content")
OUT = DRIVE / "bge-m3"
PARTS = OUT / "parts"
BLOCK = 10_000
BATCH = 32
MAX_TOKENS = 512

assert torch.cuda.is_available(), "Activer le GPU : Exécution > Modifier le type d'exécution"
print(torch.cuda.get_device_name(0))

PARTS.mkdir(parents=True, exist_ok=True)
with open(DRIVE / "chunks.jsonl", encoding="utf-8") as fh:
    texts = [json.loads(line)["text"] for line in fh]
print(f"{len(texts)} chunks")

In [ ]:
model = SentenceTransformer(MODEL, device="cuda")
model.half()
model.max_seq_length = MAX_TOKENS

t0 = time.time()
for start in range(0, len(texts), BLOCK):
    part = PARTS / f"{start:07d}.npy"
    if part.exists():
        continue
    vectors = model.encode(texts[start:start + BLOCK], batch_size=BATCH,
                           normalize_embeddings=True, show_progress_bar=True)
    np.save(part, vectors.astype(np.float32))
    print(f"{min(start + BLOCK, len(texts))}/{len(texts)} ({time.time() - t0:.0f} s)")

In [ ]:
parts = sorted(PARTS.glob("*.npy"))
vectors = np.concatenate([np.load(p) for p in parts])
assert len(vectors) == len(texts), f"{len(vectors)} vecteurs pour {len(texts)} chunks"
assert np.allclose(np.linalg.norm(vectors[:100], axis=1), 1, atol=1e-2)

np.save(OUT / "vectors.npy", vectors)
shutil.copy(DRIVE / "chunks.jsonl", OUT / "chunks.jsonl")
duration = time.time() - t0
info = {"model": MODEL, "n_chunks": len(texts), "dim": int(vectors.shape[1]),
        "duree_s": round(duration, 1), "chunks_par_s": round(len(texts) / max(duration, 1e-9), 1),
        "gpu": torch.cuda.get_device_name(0)}
(OUT / "info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(info)

In [ ]:
query = model.encode(["Hôtel pas cher à Bordeaux"], normalize_embeddings=True)[0]
with open(DRIVE / "chunks.jsonl", encoding="utf-8") as fh:
    meta = [json.loads(line) for line in fh]
for i in np.argsort(vectors @ query)[::-1][:5]:
    print(f"{vectors[i] @ query:.3f}  {meta[i]['page_title']} > {meta[i]['section']}")